## Vector Space Visualization

Our embeddings live in **1536 dimensions** — impossible to visualize directly.

**UMAP** (Uniform Manifold Approximation and Projection) compresses them to 2D/3D
while trying to preserve which dramas are *neighbours* in the original space.

If the embeddings are good, you should see **clusters** — romance dramas near each other,
historical dramas near each other, etc. That's the whole point of semantic embeddings!

In [2]:
# uv add umap-learn plotly  ← run this in terminal first if needed
import polars as pl
import numpy as np
import plotly.express as px
import umap
from pathlib import Path


DATA_PATH = Path("../data/cleaned/dramas_with_vectors.parquet")

c:\Users\danic\Documents\mdl_recommender\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load data

In [3]:
df = pl.read_parquet(DATA_PATH)
print(f"Loaded {df.height} dramas")
print(f"Embedding dimensions: {len(df['embedding'][0])}")

Loaded 1437 dramas
Embedding dimensions: 1536


### Extract embeddings as a numpy matrix

UMAP expects a 2D numpy array: shape `(n_dramas, n_dimensions)` → `(1437, 1536)`

Think of it as a spreadsheet where each row is a drama and each column is one dimension.

In [4]:
# Stack all embedding lists into a matrix
embeddings_matrix = np.array(df["embedding"].to_list())
print(f"Matrix shape: {embeddings_matrix.shape}")  # should be (1437, 1536)

Matrix shape: (1437, 1536)


### Prepare labels for colouring the plot

We'll colour by the **primary genre** (first item in the genres list).
This lets us visually check: do same-genre dramas cluster together?

In [5]:
# Extract primary genre (first in list) for colouring
# Also grab other fields for the hover tooltip
meta = df.select([
    "title",
    "year",
    "mdl_score",
    pl.col("genres").list.first().alias("primary_genre"),
    pl.col("genres").list.join(", ").alias("all_genres"),
    pl.col("tags").list.head(3).list.join(", ").alias("top_tags"),
]).to_pandas()  # plotly works with pandas

print(f"Top genres:")
print(meta["primary_genre"].value_counts().head(10))

Top genres:
primary_genre
romance       349
historical    284
comedy        211
thriller      153
action        120
mystery        74
business       63
adventure      31
life           31
wuxia          25
Name: count, dtype: int64


### Run UMAP — 2D

**Key parameters:**
- `n_components=2` → compress to 2D (use 3 for 3D)
- `n_neighbors=15` → how many nearby points to consider when building the map. 
  Lower = more local clusters, Higher = more global structure
- `min_dist=0.1` → how tightly packed clusters are. Lower = tighter clusters
- `random_state=42` → makes it reproducible (same result every run)

This takes ~30 seconds for 1437 dramas — totally normal!

In [6]:
print("Running UMAP... (takes ~30 seconds)")

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",   # same distance metric as pgvector uses!
    random_state=42,
)

coords_2d = reducer.fit_transform(embeddings_matrix)
print(f"Done! Output shape: {coords_2d.shape}")  # (1437, 2)

Running UMAP... (takes ~30 seconds)


c:\Users\danic\Documents\mdl_recommender\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Done! Output shape: (1437, 2)


### 2D Interactive Plot

Hover over any dot to see the drama details!
Colour = primary genre. Tight clusters = semantically similar dramas.

In [7]:
# Add UMAP coordinates to our metadata dataframe
meta["umap_x"] = coords_2d[:, 0]
meta["umap_y"] = coords_2d[:, 1]

# Keep only genres with enough dramas to be meaningful in the legend
top_genres = meta["primary_genre"].value_counts().head(10).index.tolist()
meta["genre_label"] = meta["primary_genre"].apply(
    lambda g: g if g in top_genres else "Other"
)

fig = px.scatter(
    meta,
    x="umap_x",
    y="umap_y",
    color="genre_label",
    hover_name="title",
    hover_data={
        "year": True,
        "mdl_score": True,
        "all_genres": True,
        "top_tags": True,
        "umap_x": False,   # hide raw coordinates from tooltip
        "umap_y": False,
    },
    title="Chinese Drama Embedding Space (UMAP 2D)",
    labels={"genre_label": "Primary Genre"},
    width=1000,
    height=700,
    opacity=0.75,
)

fig.update_traces(marker=dict(size=6))
fig.show()

### 3D Interactive Plot

Sometimes 3D reveals structure that 2D hides. You can rotate it!
UMAP already computed in 2D above, so we need to re-run for 3D.

In [8]:
print("Running UMAP 3D... (takes ~30 seconds)")

reducer_3d = umap.UMAP(
    n_components=3,    # 👈 only change from 2D version
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42,
)

coords_3d = reducer_3d.fit_transform(embeddings_matrix)

meta["umap_x3"] = coords_3d[:, 0]
meta["umap_y3"] = coords_3d[:, 1]
meta["umap_z3"] = coords_3d[:, 2]

fig3d = px.scatter_3d(
    meta,
    x="umap_x3",
    y="umap_y3",
    z="umap_z3",
    color="genre_label",
    hover_name="title",
    hover_data={
        "year": True,
        "mdl_score": True,
        "all_genres": True,
        "umap_x3": False,
        "umap_y3": False,
        "umap_z3": False,
    },
    title="Chinese Drama Embedding Space (UMAP 3D)",
    labels={"genre_label": "Primary Genre"},
    width=1000,
    height=700,
    opacity=0.75,
)

fig3d.update_traces(marker=dict(size=4))
fig3d.show()

Running UMAP 3D... (takes ~30 seconds)


c:\Users\danic\Documents\mdl_recommender\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### Sanity check: find a drama and its nearest neighbours

Pick any drama and see what's closest to it in the embedding space.
This is essentially what our recommender does!

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

# 👇 Change this to any drama you like!
DRAMA_TITLE = "How dare you!?"

# Find its index
titles = df["title"].to_list()
try:
    idx = next(i for i, t in enumerate(titles) if DRAMA_TITLE.lower() in t.lower())
except StopIteration:
    print(f"'{DRAMA_TITLE}' not found! Try another title.")
    idx = None

if idx is not None:
    query_vec = embeddings_matrix[idx].reshape(1, -1)
    sims = cosine_similarity(query_vec, embeddings_matrix)[0]
    
    # Get top 6 (index 0 will be itself with similarity 1.0)
    top_indices = sims.argsort()[::-1][1:6]
    
    print(f"\nTop 5 most similar to '{titles[idx]}':\n")
    for rank, i in enumerate(top_indices, 1):
        print(f"  {rank}. {titles[i]:<40} similarity: {sims[i]:.3f}")


Top 5 most similar to 'How Dare You!?':

  1. A Dream within a Dream                   similarity: 0.806
  2. The Princess's Gambit                    similarity: 0.798
  3. Money Is Coming                          similarity: 0.794
  4. Please Don’t Spoil Me                    similarity: 0.779
  5. Shining Just for You                     similarity: 0.779
